# DPO 
Direct Preference Optimization
不多说，直接开撕

In [1]:
# 每日一导
import torch
import torch.nn as nn
import torch.nn.functional as F


In [1]:
def seq_logprob(logits,labels,ignore_index=-100):
    """
    logits: [B, T, V]
    labels: [B, T]
    返回: [B]
    """
    # 对齐输出和标签
    logits = logits[:, :-1, :]   # 丢最后一个
    labels = labels[:, 1:] # 丢第一个

    # 转log
    log_prob = F.log_softmax(logits,dim=-1)

    # 取正确的token prob
    # 这里先把等于ignore_index的提前赋为0
    safe_labels = labels.clone()
    safe_labels[safe_labels == ignore_index] = 0
    
    # gather一下
    # 先unsqueeze
    # 再重新squeeze
    token_log_prob= log_prob.gather(
        dim=-1,
        index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)

    # mask掉不参与loss的部分
    mask = (labels!=ignore_index).float()
    token_log_prob = token_log_prob*mask

    return token_log_prob.sum(dim=-1)


In [ ]:

def get_logprob(model,input_ids,attention_mask,labels):
    """
    用某个模型计算整条回答的 log π(y|x)
    """
    outputs=model(
        input_ids=input_ids
        attention_mask=attention_mask
    )
    logits=outputs.logits
    return seq_logprob(logits,labels)

In [ ]:
def dpo_loss(
        policy_model,
        ref_model,
        chosen_input_ids,
        chosen_attention_mask,
        chosen_label,
        rejected_input_ids,
        rejected_attention_mask,
        rejected_label,
        beta=0.1,
        ignore_index=-100
):
    # 计算一个batch的DPO loss
    

    # policy model对chosen/rejected打分
    policy_chosen_logps =get_logprob(
        policy_model,
        chosen_input_ids,
        chosen_attention_mask,
        chosen_label,
        ignore_index=ignore_index,
    )
    policy_rejected_logps = get_logprob(
        policy_model,
        rejected_input_ids
        rejected_attention_mask,
        rejected_label,
        ignore_index=ignore_index
    )

    # ref model 打分
    with torch.no_grad():
        ref_chosen_logps=get_logprob(
            ref_model,
            chosen_input_ids,
            chosen_attention_mask,
            chosen_label,
            ignore_index=ignore_index,
        )
        ref_rejected_logps=get_logprob(
            ref_model,
            rejected_input_ids,
            rejected_attention_mask,
            rejected_label,
            ignore_index=ignore_index,
        )

        # 先分别做偏好差值
        policy_logratios = policy_chosen_logps-policy_rejected_logps
        ref_logratios = ref_chosen_logps-ref_rejected_logps

        # 再做KL散度惩罚
        logits = beta*(policy_logratios-ref_logratios)
        loss = -F.logsigmoid(logits).mean()

        return loss